In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets

sklearn 라이브러리가 1.9 버전이 되면서 `'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence`와 같은 에러가 발생되면 아래와 같은 코드를 실행하면 된다.

In [2]:
sklearn.set_config(display='text')

# 데이터 전처리

우리가 실제로 접하게 되는 데이터는 사이킷런 라이브러리에서 제공하는 데이터와는 다르게 머신러닝 또는 딥러닝에 적용하기 전에 데이터를 가공하는 경우가 대부분이다. 이를 데이터 전처리 또는 피처 엔지니어링 등으로 부른다.

## 결측치(missing value) 처리

데이터셋의 일부에 데이터가 존재하지 않을 경우 결측치라 부르며, 머신러닝 또는 딥러닝에 적용하기 전에 결측치를 처리하는 것이 좋다.

데이터프레임에 데이터가 존재하지 않는 결측치는 없지만 부적절한 값(이상치)이 존재한다.  
나이에 1000이라는 값이 존재하는 것과 나머치 피처에 'unknown'이라는 값이 입력된 부적절한 값은 결측치로 처리하는 것이 좋다.

In [3]:
df = pd.DataFrame([
    [42, '남', 12, '등산', 'class2'],
    [35, 'unknown', 3, '바둑', 'class1'],
    [1000, '여', 7, '낚시', 'class3'],
    [1000, 'unknown', 21, 'unknown', 'unknown'],
])

df.columns = ['나이', '성별', '태어난달', '취미', 'label']
df

,나이,성별,태어난달,취미,label
0,42,남,12,등산,class2
1,35,unknown,3,바둑,class1
2,1000,여,7,낚시,class3
3,1000,unknown,21,unknown,unknown


판다스의 unique() 메소드로 각 시리즈의 유일한 값을 확인할 수 있다.

In [4]:
print(f'나이: {df.나이.unique()}')
print(f'성별: {df.성별.unique()}')
print(f'태어난달: {df.태어난달.unique()}')
print(f'취미: {df.취미.unique()}')
print(f'label: {df.label.unique()}')

나이: [  42   35 1000]
성별: <StringArray>
['남', 'unknown', '여']
Length: 3, dtype: str
태어난달: [12  3  7 21]
취미: <StringArray>
['등산', '바둑', '낚시', 'unknown']
Length: 4, dtype: str
label: <StringArray>
['class2', 'class1', 'class3', 'unknown']
Length: 4, dtype: str


부적절한 데이터 값을 결측치로 바꾸기 위해서 np.nan를 사용한다.

In [5]:
df.loc[df.나이 > 150, '나이'] = np.nan
df.loc[df.성별 == 'unknown', '성별'] = np.nan
df.loc[df.태어난달 > 12, '태어난달'] = np.nan
df.loc[df.취미 == 'unknown', '취미'] = np.nan
df.loc[df.label == 'unknown', 'label'] = np.nan
df

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,NaN,3.0,바둑,class1
2,NaN,여,7.0,낚시,class3
3,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   나이      2 non-null      float64
 1   성별      2 non-null      str    
 2   태어난달    3 non-null      float64
 3   취미      3 non-null      str    
 4   label   3 non-null      str    
dtypes: float64(2), str(3)
memory usage: 292.0 bytes


In [7]:
df.isnull().sum()

나이       2
성별       2
태어난달     1
취미       1
label    1
dtype: int64

결측치를 처리하는 데 주로 사용되는 방법은 결측치를 삭제하거나 다른 값으로 변경하는 방법이 있다.

dropna() 메소드로 결측치를 삭제할 수 있다.  
`axis`로 결측치를 생략할 행 또는 열 방향을 지정할 수 있다. 생략하거나 0으로 지정하면 결측치를 한 개라도 포함한 모든 행을 삭제하고, 1로 지정하면 결측치가 한 개라도 포함된 모든 열을 삭제한다.

In [8]:
df.dropna(axis=0)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2


In [9]:
df.dropna(axis=1)

""
0
1
2
3


`how`를 `all`로 지정하면 모든 값이 결측치인 행만 제거한다.

In [10]:
df.dropna(how='all')

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,NaN,3.0,바둑,class1
2,NaN,여,7.0,낚시,class3


`thresh`에 숫자를 지정하면 결측치를 제외한 값의 개수가 지정한 숫자보다 작은 모든 행을 삭제한다.

In [11]:
df.dropna(thresh=4)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,NaN,3.0,바둑,class1
2,NaN,여,7.0,낚시,class3


`subset`에 열이름을 지정하면 지정한 열에 결측치가 존재하는 모든 행을 삭제한다.

In [12]:
df.dropna(subset=['성별'])

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
2,NaN,여,7.0,낚시,class3


fillna() 메소드로 결측치를 대체할 값을 지정할 수 있다.

In [13]:
df.fillna(0)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,0,3.0,바둑,class1
2,0.0,여,7.0,낚시,class3
3,0.0,0,0.0,0,0


`value`에 딕셔너리 형태로 각 열의 결측치를 대체할 값을 지정할 수 있다.

In [14]:
alter_values = {'나이': 0, '성별': 'U', '태어난달': 0, '취미': 'N', 'label': 'class4'}
df.fillna(value=alter_values, inplace=True)

,나이,성별,태어난달,취미,label
0,42.0,남,12.0,등산,class2
1,35.0,U,3.0,바둑,class1
2,0.0,여,7.0,낚시,class3
3,0.0,U,0.0,N,class4


## 기계 처리에 적합하도록 레이블을 설정한다.

label 열을 보면 문자열 형태인데, 이는 기계 처리에 적합하지 않기때문에 문자열을 숫자로 바꿔준다.

사이킷런 라이브러리를 사용하지 않고 레이블 설정하기

In [15]:
# 라벨링할 레이블을 얻어와서 오름차순으로 정렬한다.
label = list(df.label.values)
print(label)
label.sort()
# label.sort(reverse=True) # 내림차순 정렬
print(label)

['class2', 'class1', 'class3', 'class4']
['class1', 'class2', 'class3', 'class4']


In [16]:
# 반복문을 실행해서 정렬된 레이블을 key로 인덱스를 value로 하는 딕셔너리를 만든다.
label_dict = {}
for index, data in enumerate(label):
    # print(index, data)
    label_dict[data] = index
label_dict

{'class1': 0, 'class2': 1, 'class3': 2, 'class4': 3}

In [17]:
# replace() 메소드로 딕셔너리에 저장된 데이터로 레이블을 라벨링한다.
df['label2'] = df.label.replace(label_dict)
df

,나이,성별,태어난달,취미,label,label2
0,42.0,남,12.0,등산,class2,1
1,35.0,U,3.0,바둑,class1,0
2,0.0,여,7.0,낚시,class3,2
3,0.0,U,0.0,N,class4,3


사이킷런 라이브러리를 사용해서 레이블 설정하기

레이블 라벨링을 하기 위해 LabelEncoder를 import 한다.

In [18]:
from sklearn.preprocessing import LabelEncoder

In [19]:
label = list(df.label.values)
print(label)

# 레이블 라벨링을 하기 위해 LabelEncoder 객체를 만든다.
label_encoder = LabelEncoder()

# LabelEncoder 객체를 이용해서 문자열로 구성된 레이블을 기계 처리에 적합하도록 학습시키고 적용한다.
# 학습시킬 때 무조건 fit() 메소드로 라벨링할 데이터를 넘겨서 사용한다.
# label_encoder.fit(label)
# 적용시킬 때 무조건 transform() 메소드로 학습된 데이터를 적용한다.
# new_label = label_encoder.transform(label)

# 학습과 적용을 한꺼번에 실행하려면 fit_transform() 메소드를 사용한다.
new_label = label_encoder.fit_transform(label)
print(new_label)

df['label3'] = new_label
df

['class2', 'class1', 'class3', 'class4']
[1 0 2 3]


,나이,성별,태어난달,취미,label,label2,label3
0,42.0,남,12.0,등산,class2,1,1
1,35.0,U,3.0,바둑,class1,0,0
2,0.0,여,7.0,낚시,class3,2,2
3,0.0,U,0.0,N,class4,3,3


## One-Hot Encoding

원-핫 인코딩은 레이블을 라벨링하는 또 다른 방법으로 오직 False(0)와 True(1)만 사용하는 벡터를 이용해서 데이터 값을 나타내는 것이다.

판다스의 get_dummies() 메소드를 이용하는 원-핫 인코딩

`dtype`의 기본값은 bool이고 False와 True로 원-핫 인코딩을 실행하고 int를 사용하면 0과 1로 원-핫 인코딩을 실행한다.

In [20]:
pd.get_dummies(df.label)

,class1,class2,class3,class4
0,False,True,False,False
1,True,False,False,False
2,False,False,True,False
3,False,False,False,True


In [21]:
pd.get_dummies(df.label, dtype=int)

,class1,class2,class3,class4
0,0,1,0,0
1,1,0,0,0
2,0,0,1,0
3,0,0,0,1


`drop_first`의 기본값은 False이고 True를 사용하면 원-핫 인코딩 결과에서 첫 열을 제거한다.

In [22]:
pd.get_dummies(df.label, dtype=int, drop_first=True)

,class2,class3,class4
0,1,0,0
1,0,0,0
2,0,1,0
3,0,0,1


사이킷런 라이브러리를 이용하는 원-핫 인코딩

사이킷런 라이브러리를 이용하는 원-핫 인코딩을 하기 위해 OneHotEncoder를 import 한다.

In [23]:
from sklearn.preprocessing import OneHotEncoder

In [24]:
# 사이킷런 라이브러리를 이용하는 원-핫 인코딩은 원-핫 인코딩을 적용할 데이터프레임을 넘겨줘야 한다.
label = df[['label']]
print(label)

# 원-핫 인코딩을 하기 위해 OneHotEncoder 객체를 만든다.
one_hot_encoder = OneHotEncoder()

# one_hot_encoder.fit(label) # 학습시킨다.
# new_label = one_hot_encoder.transform(label) # 적용한다. => 적용 결과가 희소 행렬로 리턴된다.
new_label = one_hot_encoder.fit_transform(label)

print(new_label)
print(new_label.toarray())

    label
0  class2
1  class1
2  class3
3  class4
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (4, 4)>
  Coords	Values
  (0, 1)	1.0
  (1, 0)	1.0
  (2, 2)	1.0
  (3, 3)	1.0
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


텐서플로우 라이브러리를 이용하는 원-핫 인코딩

텐서플로우 라이브러리를 사용하기 위해 tensorflow를 설치하고 import 한다.  
`!pip install tensorflow`

tensorflow 1.x 버전을 사용하고 싶다면 아래와 같이 import 한다.  
`import tensorflow.compat.v1 as tf`  
tensorflow 2.x 버전의 문법을 사용하지 않겠다고 선언한다.  
`tf.disable_v2_behavior()`

In [25]:
import tensorflow as tf
print(tf.__version__)

2.21.0


텐서플로우 라이브러리를 이용하는 원-핫 인코딩을 하기 위해 to_categorical을 import 한다.

In [26]:
from tensorflow.keras.utils import to_categorical

In [27]:
# 텐서플로우 라이브러리를 이용하는 원-핫 인코딩은 원-핫 인코딩을 적용할 숫자로 구성된 데이터프레임을 넘겨줘야 한다.
label = df[['label2']]
print(label)

# to_categorical() 함수로 라벨링할 데이터프레임을 넘겨서 원-핫 인코딩을 실행해야 한다.
new_label = to_categorical(label)
print(new_label)

  label2
0      1
1      0
2      2
3      3
[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


## 데이터 스케일링(정규화)

피처들은 제각기 다른 단위를 가지기 때문에 이로 인해 숫자 자체의 크기 차이가 발생하므로 데이터 값이 단위의 영향을 받지 않도록 변형하는 것을 의미한다.  
`전체 데이터를 학습 데이터와 테스트 데이터로 나누고 스케일링의 기준이 되는 값은 학습 데이터로만 구한다.`

표준화(standard) 스케일링

대표적인 데이터 스케일링 방법으로 표준 정규 분포(평균이 0이고 표준편차가 1)를 따르도록 변경하는 방법이다.

$$\frac{x_i-\bar x}{\sigma}$$

위 식의 의미는 기존 데이터에서 평균을 뺀 값을 표준편차로 나눈다는 의미이다.

표준화 스케일링을 사용하기 위해 StandardScaler를 import 한다.

In [28]:
from sklearn.preprocessing i`mport StandardScaler

In [29]:
# 표준화 스케일링 객체를 만든다.
scaler = StandardScaler()

data = df[['태어난달']]
print(data)

# scaler.fit(data) # 학습
# new_scaler = scaler.transform(data) # 적용
new_scaler = scaler.fit_transform(data) # 학습 후 적용

print(type(new_scaler))
print(new_scaler)

   태어난달
0  12.0
1   3.0
2   7.0
3   0.0
<class 'numpy.ndarray'>
[[ 1.44444444]
 [-0.55555556]
 [ 0.33333333]
 [-1.22222222]]


In [30]:
print(f'평균: {new_scaler.mean():.1f}, 표준편차: {new_scaler.std():.1f}')

평균: -0.0, 표준편차: 1.0


로버스트(robust) 스케일링

표준화 스케일링을 변형한 방법으로 중위수(median)와 사분위수(quantile)를 사용한다. 극단값의 영향을 거의 받지 않는다는 장점이 있다.

$$\frac{x_i-q_2}{q_3-q_1}$$

위 식에서 $q_1$은 1사분위수, $q_2$는 2사분위수(중위수), $q_3$은 3사분위수를 의미한다.

로버스트 스케일링을 사용하기 위해 RobustScaler를 import 한다.

In [31]:
from sklearn.preprocessing import RobustScaler

In [32]:
# 로버스트 스케일링 객체를 만든다.
scaler = RobustScaler()

data = df[['태어난달']]
print(data)

# scaler.fit(data) # 학습
# new_scaler = scaler.transform(data) # 적용
new_scaler = scaler.fit_transform(data) # 학습 후 적용

print(type(new_scaler))
print(new_scaler)

   태어난달
0  12.0
1   3.0
2   7.0
3   0.0
<class 'numpy.ndarray'>
[[ 1.16666667]
 [-0.33333333]
 [ 0.33333333]
 [-0.83333333]]


최소-최대(min-max) 스케일링

데이터 값의 범위를 최소값(0)과 최대값(1) 사이의 값으로 범위를 계산한다.

$$\frac{x_i-min(x)}{max(x)-min(x)}$$

최소-최대 스케일링을 사용하기 위해 MinMaxScaler를 import 한다.

In [33]:
from sklearn.preprocessing import MinMaxScaler

In [34]:
# 최소-최대 스케일링 객체를 만든다.
scaler = MinMaxScaler()

data = df[['태어난달']]
print(data)

# scaler.fit(data) # 학습
# new_scaler = scaler.transform(data) # 적용
new_scaler = scaler.fit_transform(data) # 학습 후 적용

print(type(new_scaler))
print(new_scaler)

   태어난달
0  12.0
1   3.0
2   7.0
3   0.0
<class 'numpy.ndarray'>
[[1.        ]
 [0.25      ]
 [0.58333333]
 [0.        ]]


노멀(normalizer) 스케일링

벡터의 유클리드 거리(두 점 사이의 거리 계산에 사용됨)가 1이 되도록 데이터 값을 변경한다. 주로 벡터의 길이는 상관없고 방향(각도)만 고려할 때 사용한다.  
표준화, 로버스트, 최소-최대 스케일링은 열 기준인 것과 다르게 노멀 스케일링은 행 기준이다.

노멀 스케일링을 사용하기 위해 Normalizer를 import 한다.

In [35]:
from sklearn.preprocessing import Normalizer

In [36]:
# 노멀 스케일링 객체를 만든다.
scaler = Normalizer()

data = df[['나이', '태어난달']]
print(data)

# scaler.fit(data) # 학습
# new_scaler = scaler.transform(data) # 적용
new_scaler = scaler.fit_transform(data) # 학습 후 적용

print(type(new_scaler))
print(new_scaler)

     나이  태어난달
0  42.0  12.0
1  35.0   3.0
2   0.0   7.0
3   0.0   0.0
<class 'numpy.ndarray'>
[[0.96152395 0.27472113]
 [0.99634665 0.08540114]
 [0.         1.        ]
 [0.         0.        ]]


데이터 스케일링 과정에서 fit() 메소드는 학습 데이터에 대해서만 사용하며, 테스트 데이터에는 `fit() 메소드를 사용하지 않고 transform() 메소드만 사용`한다.  
이유는 스케일러는 학습 데이터를 기반으로 만들어져야 하기 때문이다. 만약에 학습 데이터와 테스트 데이터에 모두 fit() 메소드를 실행해서 각각의 데이터로 스케일링 한다면 학습 데이터와 테스트 데이터의 분포가 달라지기때문에 학습 데이터와 테스트 데이터의 스케일링 범위 및 파라미터가 달라진다.

`from sklearn.preprocessing import StandardScaler`  
`scaler = StandardScaler()`  

학습 데이터는  
`scaler.fit(x_train)` # 학습  
`new_scaler = scaler.transform(x_train)` # 적용  
또는  
`new_scaler = scaler.fit_transform(x_train)` # 학습 후 적용  
방법을 모두 사용할 수 있다.

테스트 데이터는 fit() 메소드를 사용하지 않고 학습 데이터로 학습된 스케일러에 테스트 데이터를 적용만 시킨다.  
`new_scaler = scaler.transform(y_train)`